# Trial {N} — <hypothesis in one line>

**Key insight:** <TBD — fill after the trial (must match cell 9 + trials.json)>

> Copy of `template.ipynb`. Read `context.md`, `instructions.md` and all previous trials before executing.
> ⚠️ NEVER "Run All" — cells 4, 5, 9 are written by the agent between execution phases.

## 1. Setup — load history + memory index

In [ ]:
import json, glob, re, os, sys

NOTEBOOK_DIR = os.getcwd()                    # this experiment folder (tpe_custom, trials.json)
REPO_ROOT = NOTEBOOK_DIR
while REPO_ROOT != os.path.dirname(REPO_ROOT) and not os.path.isdir(os.path.join(REPO_ROOT, 'src')):
    REPO_ROOT = os.path.dirname(REPO_ROOT)    # climb to repo root (dir containing src/)
sys.path.insert(0, NOTEBOOK_DIR)              # for tpe_custom / eval_harness
sys.path.insert(0, REPO_ROOT)                 # for src/
os.chdir(REPO_ROOT)                           # data/... resolve from repo root
TRIALS_PATH = os.path.join(NOTEBOOK_DIR, 'trials.json')

# --- structured history (data only, for the algorithm) ---
trials = json.load(open(TRIALS_PATH))['trials']
best = min((t for t in trials if t.get('objective') is not None),
           key=lambda t: t['objective'], default=None)
print(f"{len(trials)} trials so far | current best: {best['trial_id'] if best else None} "
      f"(obj={best['objective']:.4f})" if best else f"{len(trials)} trials so far | no best yet")

# --- memory index: Key insight of every previous trial notebook (agent's memory) ---
for p in sorted(glob.glob(os.path.join(NOTEBOOK_DIR, 'trial_*.ipynb'))):
    nb2 = json.load(open(p))
    ki = '?'
    for c in nb2['cells']:
        if c['cell_type'] == 'markdown':
            m = re.search(r'\*\*Key insight:\*\*\s*(.*)', ''.join(c['source']))
            if m: ki = m.group(1).strip()
            break
    print(f'  {os.path.basename(p)}: {ki}')

# --- algorithm + space ---
from pagho import propose, DEFAULT_SPACE
print('parameter ranges:', {k: (v['low'], v['high']) if v['type'] != 'choice' else v['values']
                             for k, v in DEFAULT_SPACE.items()})

## 2. Algorithm proposal — 10 candidates with EI scores

In [ ]:
candidates = propose(trials, space=DEFAULT_SPACE, n_candidates=10, seed=None)
for i, cand in enumerate(candidates, 1):
    cfg = cand['config']
    print(f"{i:2d}. EI={cand['score']:.3f} | " +
          '  '.join(f'{k}={v}' for k, v in cfg.items()))

## 3. Candidate analysis  (agent writes here — BEFORE executing cell 6)

- **EI ranking:** which candidates the algorithm likes (score column) and why it might.
- **Physics alignment:** which candidates attack a known failure mode (context.md) — starvation,
  low-count γ noise, KDE skewness, anneal freeze.
- **Risk:** which look dangerous (e.g., huge n_runs·n_iter → runtime, tiny lr → under-convergence,
  sigma_ref extremes) and why.
- **Patterns:** structure across the 10 (clusters, outliers, axes with little variation).
- **History:** how do these compare to previous trials (configs already tried, what worked/failed)?

## 4. Selection and reasoning  (agent writes here)

**Chosen candidate INDEX:** _

**Why this one over the others:** _

**Hypothesis (what I expect):** _

**Would confirm:** _   **Would refute:** _

## 5. Set chosen candidate — agent sets INDEX below

In [ ]:
INDEX = 1   # <-- agent sets this to the chosen candidate (1-based, from cell 3)
assert 1 <= INDEX <= len(candidates), 'bad INDEX'
CHOSEN = candidates[INDEX - 1]['config']
TRIAL_ID = 'trial_XXX'   # <-- set this when copying the template
print('chosen:', CHOSEN)

## 6. Run the trial  ⛔ ~1h — do not interrupt unless obviously broken

In [ ]:
import traceback, time
from pagho import run_trial
t0 = time.time()
try:
    results = run_trial(CHOSEN)          # synthetic benchmark, 14 exps
    print(f'trial finished in {(time.time()-t0)/60:.1f} min')
except Exception:
    traceback.print_exc()
    results = None                        # fail fast, record below

## 7. Results and objective

In [ ]:
import numpy as np
if results is not None:
    from pagho import compute_objective
    objective, uncertainty = compute_objective(results)
    print(f'objective (MSE vs true, 14 exps) = {objective:.4f} ± {uncertainty:.4f}')
    # TODO: per-exp table + plots (convergence, FWHM distributions, comparison to previous trials)
else:
    objective, uncertainty = None, None
    print('trial failed — no objective')

## 8. Post-trial reflection  (agent writes here — BEFORE running cell 10)

- **What happened** vs expectations.
- **Hypothesis confirmed or refuted** — evidence, not vibes.
- **What was learned** about the physics and the hyperparameters.
- **Patterns/insights** for future selections.
- **Next hypothesis:** what the next trial should test.

**Key insight:** <one sentence — copy into the title cell>

In [ ]:
# --- 9. Save to trials.json (run AFTER writing cell 8) ---
entry = {
    'trial_id': TRIAL_ID,
    'config': CHOSEN,
    'objective': objective,
    'uncertainty': uncertainty,
    'summary': '<one line — from the reflection>',
    'key_insight': '<same sentence as title cell>',
    'notebook': f'{TRIAL_ID}.ipynb',
}
data = json.load(open(TRIALS_PATH))
data['trials'].append(entry)
json.dump(data, open(TRIALS_PATH, 'w'), indent=2)
best = min((t for t in data['trials'] if t.get('objective') is not None),
           key=lambda t: t['objective'], default=None)
print('saved', TRIAL_ID, '| current best:', best['trial_id'] if best else None)